# Resident Trajectory Archetypes

## 1) Problem framing
**Business question:** Which cross-domain trajectory patterns indicate stronger reintegration progress?

- Explanatory goal: estimate directional effects of education/health/case activity features on **current risk level** (continuous proxy).
- Predictive goal: predict **positive trajectory** (risk improved vs intake, or positive reintegration status) using **only features that do not encode the label directly**.
- Success metrics: explanatory R2/MAE; predictive ROC-AUC / F1 with **GroupShuffleSplit by resident_id** (no resident appears in both train and test).

Deployment candidate: `/api/reports/trends/resident-trajectory` and resident risk watchlist.


## Leakage checklist

| Feature | Allowed? | Notes |
|---------|------------|-------|
| `safehouse_id`, `case_category`, `referral_source` | Yes | Static / admission context |
| Education / health / process / visitation aggregates | Yes | Historical activity totals (same-time snapshot; not future-dated) |
| **`current_risk_num`** | **Removed** | Would trivially encode components of `risk_improved` with initial risk implied |
| Label `positive_trajectory` | — | Derived from `risk_improved` OR `reint_positive`; kept out of X |

Holdout uses **20% of residents** held out entirely via `GroupShuffleSplit`.


In [ ]:

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import f1_score, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'ml-pipelines', Path.cwd().parent]:
    if (candidate / 'data_loader.py').exists():
        sys.path.insert(0, str(candidate))
        break
from data_loader import load_table

res = load_table('residents').copy()
edu = load_table('education_records').copy()
health = load_table('health_wellbeing_records').copy()
proc = load_table('process_recordings').copy()
vis = load_table('home_visitations').copy()

edu_agg = edu.groupby('resident_id').agg(attendance_mean=('attendance_rate', 'mean'), progress_mean=('progress_percent', 'mean'), edu_records=('education_record_id', 'count'))
health_agg = health.groupby('resident_id').agg(health_mean=('general_health_score', 'mean'), sleep_mean=('sleep_quality_score', 'mean'), health_records=('health_record_id', 'count'))
proc_agg = proc.groupby('resident_id').agg(process_count=('recording_id', 'count'), concerns_rate=('concerns_flagged', 'mean'), progress_rate=('progress_noted', 'mean'))
vis_agg = vis.groupby('resident_id').agg(visits=('visitation_id', 'count'), followup_rate=('follow_up_needed', 'mean'))

risk_order = {'Low': 1, 'Medium': 2, 'High': 3, 'Critical': 4}
res.loc[:, 'initial_risk_num'] = res['initial_risk_level'].map(risk_order)
res.loc[:, 'current_risk_num'] = res['current_risk_level'].map(risk_order)
res.loc[:, 'risk_improved'] = (res['current_risk_num'] < res['initial_risk_num']).astype(int)
res.loc[:, 'reint_positive'] = res['reintegration_status'].isin(['Completed', 'In Progress']).astype(int)
res.loc[:, 'positive_trajectory'] = ((res['risk_improved'] == 1) | (res['reint_positive'] == 1)).astype(int)

master = res[['resident_id', 'safehouse_id', 'case_category', 'referral_source', 'current_risk_num', 'positive_trajectory']].merge(
    edu_agg, on='resident_id', how='left').merge(health_agg, on='resident_id', how='left').merge(
    proc_agg, on='resident_id', how='left').merge(vis_agg, on='resident_id', how='left')
master = master.fillna(0)

# Exclude current_risk_num from X — it overlaps conceptually with the trajectory label
features = ['safehouse_id', 'case_category', 'referral_source', 'attendance_mean', 'progress_mean', 'edu_records', 'health_mean', 'sleep_mean', 'health_records', 'process_count', 'concerns_rate', 'progress_rate', 'visits', 'followup_rate']
X = master[features].copy()
y_reg = master['current_risk_num']
y_clf = master['positive_trajectory']
groups = master['resident_id'].values

cat_cols = [c for c in features if X[c].dtype == 'object' or str(X[c].dtype) == 'bool']
num_cols = [c for c in features if c not in cat_cols]
prep = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num_cols),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_tr, idx_te = next(gss.split(X, y_clf, groups=groups))
Xtr, Xte = X.iloc[idx_tr], X.iloc[idx_te]
ytr_reg, yte_reg = y_reg.iloc[idx_tr], y_reg.iloc[idx_te]
ytrc, ytec = y_clf.iloc[idx_tr], y_clf.iloc[idx_te]

lin = Pipeline([('prep', prep), ('model', LinearRegression())])
lin.fit(Xtr, ytr_reg)
pred_reg = lin.predict(Xte)
print('Explanatory R2:', round(r2_score(yte_reg, pred_reg), 3))
print('Explanatory MAE:', round(mean_absolute_error(yte_reg, pred_reg), 3))

baseline = Pipeline([('prep', prep), ('model', DummyClassifier(strategy='prior'))])
rf = Pipeline([('prep', prep), ('model', RandomForestClassifier(
    n_estimators=300, random_state=42, min_samples_leaf=3, class_weight='balanced_subsample'))])
baseline.fit(Xtr, ytrc)
rf.fit(Xtr, ytrc)
base_proba = baseline.predict_proba(Xte)[:, 1]
rf_proba = rf.predict_proba(Xte)[:, 1]
rf_pred = (rf_proba >= 0.5).astype(int)
print('Baseline AUC:', round(roc_auc_score(ytec, base_proba), 3))
print('RF AUC:', round(roc_auc_score(ytec, rf_proba), 3))
print('RF F1:', round(f1_score(ytec, rf_pred, zero_division=0), 3))

gkf = GroupKFold(n_splits=min(5, master['resident_id'].nunique()))
cv = cross_validate(rf, X, y_clf, cv=gkf, scoring=['roc_auc', 'f1'], groups=groups, n_jobs=-1)
print('GroupKFold CV AUC mean/std:', round(float(cv['test_roc_auc'].mean()), 3), round(float(cv['test_roc_auc'].std()), 3))

perm = permutation_importance(rf, Xte, ytec, n_repeats=8, random_state=42, scoring='roc_auc')
imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).head(10)
print('\nTop predictive drivers:')
print(imp.round(4).to_string())

coef_values = np.ravel(lin.named_steps['model'].coef_)
coef_names = lin.named_steps['prep'].get_feature_names_out()
usable = min(len(coef_values), len(coef_names))
coef = pd.Series(coef_values[:usable], index=coef_names[:usable]).sort_values(key=lambda s: s.abs(), ascending=False).head(10)
print('\nTop explanatory effects (risk level):')
print(coef.round(4).to_string())

print('\nDecision recommendations: prioritize coaching where cross-domain activity signals remain weak under group holdout validation.')


In [ ]:
import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'ml-pipelines', Path.cwd().parent]:
    if (candidate / 'trend_eval_helpers.py').exists():
        sys.path.insert(0, str(candidate))
        break
from trend_eval_helpers import print_calibration_bins, print_threshold_scan, bootstrap_linear_coefs, fairness_binary, fairness_regression_mae

print("\n=== Evaluation artifacts ===")
if ytec.nunique() >= 2:
    proba = rf.predict_proba(Xte)[:, 1]
    print_calibration_bins(ytec.values, proba)
    print_threshold_scan(ytec.values, proba)
    fairness_binary(rf, Xte, ytec, master.loc[Xte.index], ['safehouse_id', 'case_category'], min_n=8)
bootstrap_linear_coefs(lin, Xtr, ytr_reg, n_boot=150, top_k=8)
fairness_regression_mae(lin, Xte, yte_reg, master.loc[Xte.index], ['safehouse_id'], min_n=8)
